In [1]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestRegressor
from sklearn.pipeline import Pipeline
from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.metrics import mean_absolute_error, r2_score

import os
import numpy as np
import pandas as pd

import openmeteo_requests
import requests_cache
from retry_requests import retry

# -----------------------------
#  Custom transformer for daily data
# -----------------------------
class PreprocessDaily(BaseEstimator, TransformerMixin):
    def fit(self, X, y=None):
        return self
    
    def transform(self, X):
        df = X.copy()
        
        # Date features
        df['date'] = pd.to_datetime(df['date'])
        df['month'] = df['date'].dt.month
        df['day'] = df['date'].dt.day
        df['day_of_year'] = df['date'].dt.dayofyear
        df['day_of_week_code'] = df['date'].dt.dayofweek  # 0=Monday, 6=Sunday
        
        # Encode station
        df['station_code'] = df['station'].astype('category').cat.codes
        
        # Drop unused columns
        drop_cols = ['date', 'station', 'day_of_week', 'sunrise', 'sunset', 'hour']
        df = df.drop(columns=[c for c in drop_cols if c in df.columns])
        
        # Fill missing values
        df = df.fillna(0)
        
        return df

# -----------------------------
#  Load dataset
# -----------------------------
df = pd.read_csv('../datasets/raw/combined.csv')

df["overcrowding_lag1"] = df.groupby("station")["overcrowding"].shift(1)
df["overcrowding_lag7"] = df.groupby("station")["overcrowding"].shift(7)


# -----------------------------
# Prepare target
# -----------------------------
target = 'overcrowding'

X = df.drop(columns=[
    'entries',
    'exits',
    'baseline_entries',
    'baseline_exits',
    target,
])
y = df[target]

# -----------------------------
# Pipeline
# -----------------------------
pipeline = Pipeline([
    ('preprocess', PreprocessDaily()),
    ('rf', RandomForestRegressor(
        n_estimators=100,
        max_depth=15,
        min_samples_split=2,
        min_samples_leaf=3,
        random_state=42,
        n_jobs=-1,
        max_samples=0.8
    ))
])

# -----------------------------
# Train/test split
# -----------------------------
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

# -----------------------------
# Train model
# -----------------------------
import time
start = time.time()
pipeline.fit(X_train, y_train)
end = time.time()
print(f"Training time: {end - start:.2f} seconds")

# -----------------------------
# Evaluate
# -----------------------------
preds = pipeline.predict(X_test)
print("MAE:", mean_absolute_error(y_test, preds))
print("R²:", r2_score(y_test, preds))

# -----------------------------
# Feature importances
# -----------------------------
rf = pipeline.named_steps['rf']
preprocessed_X = pipeline.named_steps['preprocess'].transform(X_train)
feature_importances = pd.Series(rf.feature_importances_, index=preprocessed_X.columns)
print(feature_importances.sort_values(ascending=False).head(20))


# 7 DAY PREDICTION


# -----------------------------
#  Make sure df has dates + lags computed correctly
# -----------------------------
df["date"] = pd.to_datetime(df["date"])
df = df.sort_values(["station", "date"])

# Lags already created above, but ensure they're there:
df["overcrowding_lag1"] = df.groupby("station")["overcrowding"].shift(1)
df["overcrowding_lag7"] = df.groupby("station")["overcrowding"].shift(7)

# We'll use last available state for each station
latest = df.groupby("station").tail(1).copy()

# -----------------------------
#  Fetch 7-day DAILY weather forecast (Open-Meteo) using your coords
# -----------------------------
LAT, LON = 51.4673, -0.4529

cache_session = requests_cache.CachedSession(".cache", expire_after=3600)
retry_session = retry(cache_session, retries=5, backoff_factor=0.2)
openmeteo = openmeteo_requests.Client(session=retry_session)

url = "https://api.open-meteo.com/v1/forecast"
params = {
    "latitude": LAT,
    "longitude": LON,
    "daily": ["temperature_2m_mean", "precipitation_sum"],
    "forecast_days": 7,
    "timezone": "Europe/London",
}
responses = openmeteo.weather_api(url, params=params)
response = responses[0]

daily = response.Daily()
dates = pd.date_range(
    start=pd.to_datetime(daily.Time(), unit="s"),
    end=pd.to_datetime(daily.TimeEnd(), unit="s"),
    freq=pd.Timedelta(seconds=daily.Interval()),
    inclusive="left",
)

weather_7d = pd.DataFrame({
    "date": pd.to_datetime(dates),
    "temp_mean": daily.Variables(0).ValuesAsNumpy().astype(float),
    "precip_mm": daily.Variables(1).ValuesAsNumpy().astype(float),
})
weather_7d["is_raining"] = (weather_7d["precip_mm"] > 0).astype(int)

os.makedirs("data/processed", exist_ok=True)
weather_7d.to_csv("data/processed/weather_forecast_7d.csv", index=False)

# -----------------------------
#  Build future rows and do rolling predictions per station
#     IMPORTANT: we must update lag1 each day with yesterday's prediction
# -----------------------------
# Identify the EXACT feature columns your pipeline expects
# (matches how you built X earlier)
X_cols = [c for c in df.columns if c not in ["entries", "exits", "baseline_entries", "baseline_exits", "overcrowding"]]

# Lock station categories so station_code stays consistent inside PreprocessDaily
station_categories = pd.Categorical(df["station"]).categories

outputs = []

for station, last_row in latest.set_index("station").iterrows():
    # Starting lags:
    prev1 = float(last_row["overcrowding"])  # most recent actual overcrowding
    prev7 = float(last_row["overcrowding_lag7"]) if pd.notna(last_row["overcrowding_lag7"]) else prev1

    for _, w in weather_7d.iterrows():
        future_date = w["date"]

        # Create a future row by copying last known row as a template
        row = last_row.copy()
        row["date"] = future_date
        row["station"] = station

        # Update lag features for the forecast step
        row["overcrowding_lag1"] = prev1
        row["overcrowding_lag7"] = prev7

        # Overwrite weather fields IF they exist in your dataset
        if "temp_mean" in row.index:
            row["temp_mean"] = float(w["temp_mean"])
        if "precip_mm" in row.index:
            row["precip_mm"] = float(w["precip_mm"])
        if "is_raining" in row.index:
            row["is_raining"] = int(w["is_raining"])
        # Some datasets use rain_mm too
        if "rain_mm" in row.index:
            row["rain_mm"] = float(w["precip_mm"])

        # Ensure hour exists if present (PreprocessDaily drops it anyway)
        if "hour" in row.index:
            row["hour"] = 0

        # Build X row (DataFrame)
        X_future = pd.DataFrame([row[X_cols].to_dict()])

        # Lock station categories for consistent cat.codes
        if "station" in X_future.columns:
            X_future["station"] = pd.Categorical(X_future["station"], categories=station_categories)

        # Predict
        pred = float(pipeline.predict(X_future)[0])

        outputs.append({
            "station": station,
            "date": future_date.date(),
            "predicted_overcrowding": pred,
        })

        # Roll forward lags
        prev7 = prev1
        prev1 = pred

forecast = pd.DataFrame(outputs)

# -----------------------------
# 9D) Add "normal vs predicted" + risk labels for UI
# -----------------------------
df["day_of_week_code"] = df["date"].dt.dayofweek
df["overcrowding_clipped"] = df["overcrowding"].clip(lower=0, upper=100)
baseline = (
    df.groupby(["station", "day_of_week_code"])["overcrowding_clipped"]
      .median()
      .reset_index()
      .rename(columns={"overcrowding_clipped": "baseline_overcrowding"})
)

forecast["day_of_week_code"] = pd.to_datetime(forecast["date"]).dt.dayofweek
forecast = forecast.merge(baseline, on=["station", "day_of_week_code"], how="left")
forecast["delta"] = forecast["predicted_overcrowding"] - forecast["baseline_overcrowding"]

def risk(delta):
    if delta > 8: return "HIGH"
    if delta > 3: return "MED"
    return "LOW"

forecast["risk_label"] = forecast["delta"].apply(risk)
forecast["reason"] = np.where(
    forecast["risk_label"] == "HIGH",
    "Above normal + momentum/weather",
    np.where(forecast["risk_label"] == "MED", "Slightly above normal", "Within normal range")
)

forecast = forecast.sort_values(["station", "date"])
forecast.to_csv("data/processed/forecast.csv", index=False)

forecast.head(100)

Training time: 25.12 seconds
MAE: 7.486760882121887
R²: 0.47373533536553214
overcrowding_lag1    0.473134
day_of_week_code     0.098670
overcrowding_lag7    0.097478
daylight_s           0.065645
station_code         0.040890
day                  0.036903
day_of_year          0.036579
wind_dir             0.023743
sunshine_s           0.019711
wind_gust_max        0.017254
wind_max             0.014010
temp_min             0.009593
temp_max             0.009200
app_temp_max         0.008772
app_temp_min         0.008628
precip_hours         0.008555
temp_mean            0.007198
precip_mm            0.006490
app_temp_mean        0.006377
rain_mm              0.006257
dtype: float64


,station,date,predicted_overcrowding,day_of_week_code,baseline_overcrowding,delta,risk_label,reason
0,Abbey Road DLR,2026-02-08,87.396510,6,100.000000,-12.603490,LOW,Within normal range
1,Abbey Road DLR,2026-02-09,92.157684,0,100.000000,-7.842316,LOW,Within normal range
2,Abbey Road DLR,2026-02-10,93.391249,1,100.000000,-6.608751,LOW,Within normal range
3,Abbey Road DLR,2026-02-11,92.945810,2,100.000000,-7.054190,LOW,Within normal range
4,Abbey Road DLR,2026-02-12,93.523063,3,100.000000,-6.476937,LOW,Within normal range
...,...,...,...,...,...,...,...,...
95,Arnos Grove,2026-02-12,95.590605,3,100.000000,-4.409395,LOW,Within normal range
96,Arnos Grove,2026-02-13,93.674292,4,100.000000,-6.325708,LOW,Within normal range
97,Arnos Grove,2026-02-14,91.091197,5,100.000000,-8.908803,LOW,Within normal range
98,Arsenal,2026-02-08,120.203396,6,64.217187,55.986209,HIGH,Above normal + momentum/weather
